In [47]:
from src.utils import get_path, set_device, find_closest_power_of_2, get_diffusion_schedule_nickname
from src.MSQDDPM_angel import MSQDDPM, WassDistance
import torch
import numpy as np
import matplotlib.pyplot as plt
from functools import partial
from tqdm import tqdm
import yaml
from pathlib import Path

# Actually read the config
modeldir = Path('results_UNet/CLUSTER0_5/modelresults_linear-slope8_N500_M32_n5_T20')
with open(modeldir/'config.yaml') as f:
    config = yaml.safe_load(f)

n_qubits = int(config["dataset"]["name"].split('_')[1])
n_features = 2**n_qubits

n_data = config['dataset']['maxsize']
n_timesteps = config['model']['n_timesteps']
diffusion_schedule = get_diffusion_schedule_nickname(config)
_, n_qubits = find_closest_power_of_2(n_features, return_power=True)

device = set_device(config.get('device', 'cpu'))
torch.set_default_device(device)
get_path = partial(get_path, config, modeltype='UNet', diffusion_schedule_nickname=diffusion_schedule, n_data=n_data, n_features=n_features, n_qubits=n_qubits, n_timesteps=n_timesteps)
n_limit=0

Using device: cuda:0


In [48]:
# Load the model
from diffusers import UNet1DModel
model = UNet1DModel(sample_size=n_features, **config['model']['unet_config']).to(device)
checkpoint = torch.load(modeldir / 'bestresults/bestmodel.pt', map_location=device)
model_weights = checkpoint['model_state_dict']
model.load_state_dict(model_weights)
model.to(device)
None

/tmp/ipykernel_3336726/3551147254.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(modeldir / 'bestresults/bestmodel.pt', map_location=device)


In [ ]:
# Compute Wasserstein distance across timesteps
torch.set_default_device(device)

dir, filename = get_path(type='diffusedqstates.npy')
states_diffused = np.load(dir/filename)
print(f"Diffused states shape: {states_diffused.shape}")

# Compute the Wasserstein Distance
inputs_last_timestep = torch.from_numpy(states_diffused[-1]).to(device)
wass = np.zeros(n_timesteps+1)
model.eval()
with torch.no_grad():
    inputs_last_timestep = torch.view_as_real(inputs_last_timestep) # [T+1, n_data, n_features (real), 2]
    inputs_last_timestep = inputs_last_timestep.permute(0, 1, 3, 2) # [T+1, n_data, n_channels=2, n_features]
        
    inputs_tplus1 = inputs_last_timestep
    inputs_tplus1.to(device)
    for t in tqdm(range(n_timesteps+1, 0, -1)):
        # t = torch.tensor(t).to(device)
        if t < n_limit:
            continue
        print(inputs_tplus1.shape)
        outputs = model(inputs_tplus1, t, return_dict=False)
        print(states_diffused[0].shape, outputs.shape)
        wass[t] = WassDistance(states_diffused[0], outputs).detach().cpu().numpy()
        inputs_tplus1 = outputs
    
dir, filename = get_path(type='wassdistbackwardtrain.npy')
np.save(dir / filename, wass)

# Now of the forward pass
# Wasserstein distance
wass = np.zeros(n_timesteps+1)
for t in tqdm(range(n_timesteps+1)):
    if t < n_limit:
        continue
    np.random.seed()
    wass[t] = WassDistance(torch.from_numpy(states_diffused[0]).to(device), torch.from_numpy(states_diffused[t]).to(device)).detach().numpy()

dir, filename = get_path(type='wassdistforward.npy')
np.save(dir / filename, wass)

Diffused states shape: (21, 500, 32)


  0%|          | 0/21 [00:00<?, ?it/s]

torch.Size([500, 32])


IndexError: tuple index out of range